# 03 — DenseNet121 Pretrained Baseline

Mục tiêu: lấy **điểm validation thật đầu tiên** trên protocol đã khóa.

- Dùng `split_outputs/split_manifest.csv`.
- Mặc định chạy **fold 0** để có baseline nhanh.
- **Không đụng internal holdout**.
- Backbone: DenseNet121 pretrained ImageNet.
- Positive class: `Pneumonia = 1`.
- Model selection dùng official competition score tại threshold 0.50; threshold tuning chỉ báo cáo sau khi best epoch đã được chọn.


## 0. Trước khi chạy

Nếu cell import báo thiếu thư viện, mở Terminal của VS Code và cài:

`python -m pip install torch torchvision scikit-learn pandas numpy matplotlib pillow`

Nếu `torch.cuda.is_available()` là `False`, notebook vẫn chạy nhưng DenseNet121 trên CPU sẽ chậm đáng kể.

In [ ]:
from pathlib import Path
import json, time, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

SEED = 2026
PROJECT_ROOT = Path(r'D:\medicine')
MANIFEST_PATH = PROJECT_ROOT/'split_outputs'/'split_manifest.csv'
OUTPUT_DIR = PROJECT_ROOT/'baseline_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

FOLD = 0
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 8
PATIENCE = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
print('Manifest:', MANIFEST_PATH, '| exists=', MANIFEST_PATH.exists())


## 1. Load split manifest — holdout remains sealed

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
dev = manifest[manifest['split']=='dev'].copy()
holdout = manifest[manifest['split']=='holdout'].copy()

train_df = dev[dev['cv_fold'] != FOLD].copy().reset_index(drop=True)
val_df = dev[dev['cv_fold'] == FOLD].copy().reset_index(drop=True)

assert len(train_df) == 3489, f'Unexpected train size: {len(train_df)}'
assert len(val_df) == 873, f'Unexpected val size: {len(val_df)}'
assert set(train_df['sha256']).isdisjoint(set(val_df['sha256']))
assert set(train_df['sha256']).isdisjoint(set(holdout['sha256']))
assert set(val_df['sha256']).isdisjoint(set(holdout['sha256']))

print('Train:', len(train_df), train_df['label'].value_counts().to_dict())
print('Validation:', len(val_df), val_df['label'].value_counts().to_dict())
print('SEALED holdout:', len(holdout), holdout['label'].value_counts().to_dict())
print('Leakage checks: PASS')


## 2. Transforms

Không crop mạnh. Ảnh được pad thành hình vuông trước khi resize để giữ toàn bộ trường phổi. Augmentation nhẹ và **không horizontal flip**.

In [ ]:
class PadToSquare:
    def __call__(self, img):
        w,h = img.size
        side = max(w,h)
        left = (side-w)//2; right = side-w-left
        top = (side-h)//2; bottom = side-h-top
        return ImageOps.expand(img, border=(left,top,right,bottom), fill=0)

IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD = [0.229,0.224,0.225]

train_transform = transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomAffine(degrees=5, translate=(0.02,0.02), scale=(0.95,1.05), fill=0),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
LABEL_TO_INT = {'Normal':0,'Pneumonia':1}

class XrayDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row['path']) as img:
            image = img.convert('RGB')
        return self.transform(image), LABEL_TO_INT[row['label']], row['filename']

train_ds = XrayDataset(train_df, train_transform)
val_ds = XrayDataset(val_df, val_transform)
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP)

images, labels, names = next(iter(train_loader))
print('Batch:', images.shape, labels.shape, names[:2])


## 3. Official metric

In [ ]:
def competition_metrics(y_true, probs, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    pred = (probs >= threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    acc = (tp+tn)/max(tp+tn+fp+fn,1)
    se = tp/max(tp+fn,1)
    sp = tn/max(tn+fp,1)
    f1 = f1_score(y_true,pred,pos_label=1,zero_division=0)
    P = 0.30*acc + 0.25*se + 0.20*sp + 0.25*f1
    try: auc = roc_auc_score(y_true,probs)
    except ValueError: auc = float('nan')
    return {'P':P,'Accuracy':acc,'Sensitivity':se,'Specificity':sp,'F1':f1,'AUROC':auc,'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'threshold':float(threshold)}

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys=[]; ps=[]; files=[]
    for x,y,names in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x)
        probs = torch.softmax(logits.float(),dim=1)[:,1]
        ys.extend(y.numpy().tolist()); ps.extend(probs.cpu().numpy().tolist()); files.extend(list(names))
    return np.array(ys), np.array(ps), files


## 4. DenseNet121 pretrained ImageNet + class-weighted loss

In [ ]:
weights = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=weights)
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(DEVICE)

train_counts = train_df['label'].value_counts()
n0 = int(train_counts['Normal']); n1 = int(train_counts['Pneumonia']); total = n0+n1
class_weights = torch.tensor([total/(2*n0), total/(2*n1)],dtype=torch.float32,device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print('Class weights [Normal, Pneumonia]:',class_weights.detach().cpu().numpy())
print('Trainable parameters:',f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## 5. Train fold 0

Best epoch được chọn bằng official `P` tại threshold **0.50** để tránh threshold và epoch cùng overfit một validation fold.

In [ ]:
history=[]; best_score=-1; best_epoch=0; best_state=None; stale=0

for epoch in range(1,EPOCHS+1):
    started=time.time(); model.train(); running=0.0; seen=0
    for x,y,_ in train_loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits=model(x); loss=criterion(logits,y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running += float(loss.item())*len(x); seen += len(x)
    scheduler.step()

    yv,pv,_ = predict(model,val_loader)
    m = competition_metrics(yv,pv,0.5)
    train_loss = running/max(seen,1)
    history.append({'epoch':epoch,'train_loss':train_loss,**m})
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | P={m['P']:.4f} | A={m['Accuracy']:.4f} | Se={m['Sensitivity']:.4f} | Sp={m['Specificity']:.4f} | F1={m['F1']:.4f} | {time.time()-started:.0f}s")

    if m['P'] > best_score + 1e-6:
        best_score=m['P']; best_epoch=epoch; stale=0
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        stale += 1
        if stale >= PATIENCE:
            print('Early stopping.'); break

if best_state is None: raise RuntimeError('No valid checkpoint produced.')
model.load_state_dict(best_state)
print('Best epoch:',best_epoch,'| P@0.50=',round(best_score,6))


## 6. Validation report + threshold scan

In [ ]:
yv,pv,val_names = predict(model,val_loader)
fixed = competition_metrics(yv,pv,0.5)

thresholds = np.round(np.arange(0.20,0.801,0.005),3)
rows=[competition_metrics(yv,pv,t) for t in thresholds]
threshold_table=pd.DataFrame(rows)
best_row=threshold_table.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
best_threshold=float(best_row['threshold'])
tuned=competition_metrics(yv,pv,best_threshold)

print('P@0.50:',json.dumps(fixed,indent=2))
print('Best single-fold threshold:',best_threshold)
print('Tuned:',json.dumps(tuned,indent=2))
print('NOTE: threshold này chưa phải final; final threshold phải ổn định qua 5 folds.')

cm=np.array([[tuned['TN'],tuned['FP']],[tuned['FN'],tuned['TP']]])
fig,ax=plt.subplots(figsize=(5,4)); ax.imshow(cm,cmap='Blues')
ax.set_xticks([0,1],['Normal','Pneumonia']); ax.set_yticks([0,1],['Normal','Pneumonia'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Fold {FOLD} confusion matrix @ {best_threshold:.3f}')
for r in range(2):
    for c in range(2): ax.text(c,r,str(cm[r,c]),ha='center',va='center')
plt.tight_layout(); plt.show()


## 7. Save baseline artifacts

In [ ]:
history_df=pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR/f'fold{FOLD}_history.csv',index=False)
threshold_table.to_csv(OUTPUT_DIR/f'fold{FOLD}_threshold_scan.csv',index=False)
pred_df=pd.DataFrame({'filename':val_names,'y_true':yv,'p_pneumonia':pv})
pred_df.to_csv(OUTPUT_DIR/f'fold{FOLD}_validation_predictions.csv',index=False)
torch.save({'architecture':'densenet121','weights':'ImageNet DEFAULT','fold':FOLD,'state_dict':best_state,'image_size':IMAGE_SIZE,'best_epoch':best_epoch,'P_at_0_5':fixed,'single_fold_best_threshold':best_threshold,'single_fold_tuned_metrics':tuned,'seed':SEED},OUTPUT_DIR/f'densenet121_fold{FOLD}.pt')

summary={'model':'DenseNet121','pretrained':'ImageNet DEFAULT','fold':FOLD,'train_images':len(train_df),'val_images':len(val_df),'best_epoch':best_epoch,'metrics_at_0_5':fixed,'single_fold_best_threshold':best_threshold,'single_fold_tuned_metrics':tuned,'holdout_used':False}
with open(OUTPUT_DIR/f'fold{FOLD}_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print('Saved to:',OUTPUT_DIR)
